In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os 
if os.environ.get('OPENAI_API_KEY', None) is None or os.environ.get('GEMINI_API_KEY', None) is None or os.environ.get('TAVILY_API_KEY', None) is None: 
    raise RuntimeError("Missing API keys.")

In [3]:
# Run Kestra from subprecess (allows exporting keys directly)

import subprocess, os

env = os.environ.copy()
env["SECRET_GEMINI_API_KEY"] = subprocess.check_output(
    f"echo -n {env['GEMINI_API_KEY']} | base64", shell=True
).decode().strip()

env["SECRET_OPENAI_API_KEY"] = subprocess.check_output(
    f"echo -n {env['OPENAI_API_KEY']} | base64", shell=True
).decode().strip()

env["SECRET_TAVILY_API_KEY"] = subprocess.check_output(
    f"echo -n {env['TAVILY_API_KEY']} | base64", shell=True
).decode().strip()

subprocess.run(["docker", "compose", "up", "-d"], env=env)

 kestra Pulling 
 kestra Pulled 
 Container homework_3_orchestration-kestra_postgres-1  Running
 Container homework_3_orchestration-kestra-1  Running


CompletedProcess(args=['docker', 'compose', 'up', '-d'], returncode=0)

In [4]:
# Make sure the container is running 
!docker ps

CONTAINER ID   IMAGE                   COMMAND                  CREATED          STATUS                             PORTS                                                             NAMES
5a800c7f6f45   kestra/kestra:v1.3.21   "docker-entrypoint.s…"   23 seconds ago   Up 22 seconds                      0.0.0.0:8080-8081->8080-8081/tcp, [::]:8080-8081->8080-8081/tcp   homework_3_orchestration-kestra-1
6bef1bc9629f   postgres:18             "docker-entrypoint.s…"   23 seconds ago   Up 22 seconds (health: starting)   5432/tcp                                                          homework_3_orchestration-kestra_postgres-1


Open http://localhost:8080 on a browser and login with original e-mail and password `admin@kestra.io` and `Admin1234!`

In [ ]:
!curl -X POST -u 'admin@kestra.io:Admin1234!' http://localhost:8080/api/v1/flows/import -F fileUpload=@flows/1_chat_without_rag.yaml
!curl -X POST -u 'admin@kestra.io:Admin1234!' http://localhost:8080/api/v1/flows/import -F fileUpload=@flows/2_chat_with_rag.yaml
!curl -X POST -u 'admin@kestra.io:Admin1234!' http://localhost:8080/api/v1/flows/import -F fileUpload=@flows/4_simple_agent.yaml
!curl -X POST -u 'admin@kestra.io:Admin1234!' http://localhost:8080/api/v1/flows/import -F fileUpload=@flows/5_web_research_agent.yaml
# !curl -X POST -u 'admin@kestra.io:Admin1234!' http://localhost:8080/api/v1/flows/import -F fileUpload=@flows/6_multi_agent_research.yaml

[][][][][]

### Q1. Context Engineering

ChatGPT reply:

```{yaml}
id: nyc_taxi_csv_to_bigquery
namespace: examples.data

description: Load NYC Taxi CSV data into BigQuery

inputs:
  - id: taxi_year
    type: STRING
    defaults: "2024"

  - id: taxi_month
    type: STRING
    defaults: "01"

tasks:
  - id: download_csv
    type: io.kestra.plugin.core.http.Download
    uri: https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_{{ inputs.taxi_year }}-{{ inputs.taxi_month }}.csv.gz

  - id: upload_to_gcs
    type: io.kestra.plugin.gcp.gcs.Upload
    serviceAccount: "{{ secret('GCP_SERVICE_ACCOUNT') }}"
    bucket: nyc-taxi-data
    from: "{{ outputs.download_csv.uri }}"
    to: taxi/yellow_tripdata_{{ inputs.taxi_year }}_{{ inputs.taxi_month }}.csv.gz

  - id: create_dataset
    type: io.kestra.plugin.gcp.bigquery.CreateDataset
    serviceAccount: "{{ secret('GCP_SERVICE_ACCOUNT') }}"
    datasetId: nyc_taxi

  - id: load_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.Load
    serviceAccount: "{{ secret('GCP_SERVICE_ACCOUNT') }}"
    projectId: my-gcp-project
    datasetId: nyc_taxi
    tableId: yellow_trips
    gcsUris:
      - gs://nyc-taxi-data/taxi/yellow_tripdata_{{ inputs.taxi_year }}_{{ inputs.taxi_month }}.csv.gz

    format: CSV
    autodetect: true

    writeDisposition: WRITE_APPEND

    csvOptions:
      skipLeadingRows: 1

  - id: query_sample
    type: io.kestra.plugin.gcp.bigquery.Query
    serviceAccount: "{{ secret('GCP_SERVICE_ACCOUNT') }}"
    projectId: my-gcp-project

    sql: |
      SELECT
        COUNT(*) AS total_trips
      FROM
        `my-gcp-project.nyc_taxi.yellow_trips`

    store: true
```

Kestra's AI Copilot reply: 
```{yaml}
id: nyc-taxi-data-to-bigquery
namespace: dev.nyc

description: "Loads NYC taxi data from a CSV file, uploads it to Google Cloud Storage, and then loads it into BigQuery."

inputs:
  - id: nyc_taxi_data_csv
    type: FILE
    description: "The NYC taxi data CSV file to be processed."

tasks:
  - id: upload_csv_to_gcs
    type: io.kestra.plugin.gcp.gcs.Upload
    description: "Uploads the provided CSV file to Google Cloud Storage."
    from: "{{ inputs.nyc_taxi_data_csv }}"
    to: "gs://your-gcs-bucket/nyc_taxi_data/{{ flow.id }}/{{ execution.id }}/nyc_taxi_data.csv"
    projectId: "your-gcp-project-id" # ! IMPORTANT: Replace with your GCP Project ID

  - id: load_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.LoadFromGcs
    description: "Loads the CSV data from GCS into a BigQuery table."
    from:
      - "{{ outputs.upload_csv_to_gcs.uri }}"
    destinationTable: "your_project_id.your_dataset_id.nyc_taxi_data" # ! IMPORTANT: Replace with your BigQuery Project, Dataset, and Table IDs
    format: CSV
    autodetect: true # Set to true to let BigQuery infer the schema
    csvOptions:
      skipLeadingRows: 1 # Assuming the CSV has a header row
      allowJaggedRows: true
      allowQuotedNewLines: true
    projectId: "your-gcp-project-id" # ! IMPORTANT: Replace with your GCP Project ID
```

**Answer**. Kestra's AI copilot is more flexible and context-aware. It doesn't include the data downloading step, as ChatGPT-generated flow did, but it is possible to "customize" and add steps by giving further instructions to the autopilot. 

The reason why Kestra's copilot generates better flows is due to the fact that it has access to the plugin documentation directly, providing flows with the most up-to-date plugin versions and practices. Bare ChatGPT, on the other hands, was trained on data up to a specific cutoff date, which can make the model's information obsolete in the medium/long term. Note that none of the other options is true: AI copilot can use the same model as ChatGPT (or Gemini, or any other of the supported models), it uses about the same number of tokens (if not less) for the same prompt, and it has the same level of internet access as the model we are using to query the copilot. 

The answer is therefore `AI Copilot has access to current Kestra plugin documentation`. 

### Q2. RAG Comparison

#### Chat without RAG's answer: 


Kestra 1.1 introduced several exciting features that significantly enhanced its capabilities and user experience. Here are at least 5 major features with brief descriptions:

1.  **Retry on Failures:**
    *   **Description:** This feature allows users to configure automatic retries for failed tasks or flows. Instead of immediately marking a task as failed, Kestra can now attempt to re-execute it a specified number of times, often with configurable delays between retries. This is crucial for handling transient errors (e.g., network glitches, temporary service unavailability) and improving the robustness of workflows without manual intervention.

2.  **Webhooks for External Triggers & Notifications:**
    *   **Description:** Kestra 1.1 added robust support for webhooks. This enables two key functionalities:
        *   **External Triggers:** Workflows can now be triggered by events happening in external systems that send HTTP POST requests (webhooks) to Kestra.
        *   **Notifications:** Kestra can send webhook notifications to external systems upon specific events in a flow (e.g., completion, failure), allowing for integration with custom alerting systems, monitoring tools, or other services.

3.  **Advanced Outputs (Complex Objects & Structures):**
    *   **Description:** Previously, task outputs were primarily simple strings or basic values. Kestra 1.1 significantly enhanced the output handling, allowing tasks to produce and propagate complex data structures, such as JSON objects, arrays, or even entire files. This makes it much easier to pass rich contextual information between tasks and allows for more sophisticated data processing within workflows.

4.  **Version Management & Rollback for Flows:**
    *   **Description:** This feature introduced the ability to track and manage different versions of a workflow definition. Users can now view a history of changes to their flows, revert to previous versions if needed, and ensure a stable and auditable development process. This is invaluable for collaboration, debugging, and maintaining the integrity of production workflows.

5.  **Simplified Development with New Dev Tools & CLI Enhancements:**
    *   **Description:** Kestra 1.1 included several quality-of-life improvements for developers. This involved new command-line interface (CLI) commands, better local development tools, and potentially enhanced debugging capabilities. The goal was to streamline the process of creating, testing, and deploying Kestra workflows, making it more efficient for engineers to work with the platform.

These features collectively made Kestra 1.1 a much more powerful, resilient, and developer-friendly orchestration platform.

#### Chat with RAG's answer: 

Here are 5 major features released in Kestra 1.1:

1.  **New Filters**: The UI filters have been completely redesigned for improved usability, offering explicit filter options, a single-click reset, the ability to save frequently used filter combinations, and options to hide, show, or reorder table columns.
2.  **No-Code Dashboard Editor**: This feature allows users to create and edit custom dashboards using a no-code, multi-panel editor directly from the UI, similar to the experience for flows. Users can build and customize dashboards without writing YAML.
3.  **Multi-Agent AI Systems**: AI agents can now use other AI agents as tools, enabling the orchestration of sophisticated multi-agent workflows where a primary agent can delegate subtasks to specialized expert agents.
4.  **Fix with AI**: Kestra 1.1 provides AI-powered suggestions to help diagnose and resolve issues when task runs fail, offering intelligent recommendations to troubleshoot workflow problems faster.
5.  **Human Task**: This new feature, available in the Enterprise Edition, allows workflows to pause at a designated step until specific users or group members manually approve and resume the execution, enabling human-in-the-loop workflows for critical operations.

**Answer**. The RAG version provides details, accurate features detailed in the documentation. This is because the example document ingested for the RAG step (https://raw.githubusercontent.com/kestra-io/docs/refs/heads/main/src/contents/blogs/release-1-1/index.md), contains the list and explanation of the specific features introduced in Kestra 1.1 and so the model has a precise reference to answer the question. Not only that, the model appears also more "confident" by giving shorter but more accurate answers. Without RAG, on the other hand, answers appear to be more "mechanical" and "memorized", probably they appear in the model's training data, but lacks specificity and/or confidence. 

nswer: `Vague, generic, or fabricated - the model guesses from training data`. 

### Q3. Token Usage - short summary  & Q4. Token Usage - long summary

Multilingual Agent Task usage:
- short  
    - input token count: 282 
    - output token count: 77 
    - total token count: 359
- long 
    - input token count: 282
    - output token count: 204 
    - total token count: 486

In [7]:
short_total_token_count = 359
long_total_token_count = 475 

short_output_token_count = 77 
long_output_token_count = 193

In [8]:
print(f"Q3 Answer: {short_output_token_count} tokens generated by the short summary agent.")

Q3 Answer: 77 tokens generated by the short summary agent.


Q3 Answer: `60-100 tokens`

In [9]:
print(f"Q4 Answer: While the total token count with the long summary is just {long_total_token_count/short_total_token_count:.2f}x larger, ",
      f"the output token count (which is where the two executions differed) was {long_output_token_count/short_output_token_count:.2f}x larger using the long summary.")

Q4 Answer: While the total token count with the long summary is just 1.32x larger,  the output token count (which is where the two executions differed) was 2.51x larger using the long summary.


Q4 Answer: `Long summary uses 2-4x more output tokens than short summary`

### Q5. Modifying a flow

English brevity task modified to: 

```
- id: english_brevity
    type: io.kestra.plugin.ai.agent.AIAgent
    prompt: |
      Generate exactly 3 sentences English summary of the following:
      "{{ outputs.multilingual_agent.textOutput }}"
```

Output English brevity agent output tokens usage - 3 sentences - Long Summary: 178 <br>
Output English breviy agent output tokens usage - 1 sentence - Long Summary: 46 

In [13]:
print(f"Answer question 5: the 3-sentences English Brevity agent produces {(178 / 46):.2f}x more output tokens as the 1-sentence one.")

Answer question 5: the 3-sentences English Brevity agent produces 3.87x more output tokens as the 1-sentence one.


Answer Q5: `2-4x more`

### Q6. Best Practices 

While agentic systems are more flexible than traditional workflows and can ensure up-to-date information using e.g. web search tools, they can introduce non-determinism and unexpected results which might results in non-compliance and non-repetability. Traditional, deterministic task-based workflows, on the other hands, are less flexible but more fit for strict compliance, audit, and repetability of the results. The answer is therefore `Use traditional task-based workflows for predictability and auditability`.  

In [14]:
# Close Kestra
subprocess.run(["docker", "compose", "down"], env=env)

 Container homework_3_orchestration-kestra-1  Stopping
 Container homework_3_orchestration-kestra-1  Stopped
 Container homework_3_orchestration-kestra-1  Removing
 Container homework_3_orchestration-kestra-1  Removed
 Container homework_3_orchestration-kestra_postgres-1  Stopping
 Container homework_3_orchestration-kestra_postgres-1  Stopped
 Container homework_3_orchestration-kestra_postgres-1  Removing
 Container homework_3_orchestration-kestra_postgres-1  Removed
 Network homework_3_orchestration_default  Removing
 Network homework_3_orchestration_default  Removed


CompletedProcess(args=['docker', 'compose', 'down'], returncode=0)